# 00 · Setup & environment check

Welcome to the Monster Lab workbook. Before anything else, this notebook proves your
environment works: it connects to the project database, lists what's in it, and renders
one plot. **This notebook is complete code — just run every cell top to bottom.**

Everything after this one is an exercise. Each exercise is three cells:

1. **Task** (markdown) — what to produce, and the exact variable name(s) to produce it in.
2. **Work** (code) — a skeleton with `...` placeholders. Your code replaces the `...`.
3. **Check** (code) — asserts against known-good values. It tells you *what* is wrong,
   never *how* to fix it. A passing check prints confirmation plus one sentence of context.

Read `workbook/README.md` for the rules of engagement before you start.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn

print(f"python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"scipy       {scipy.__version__}")
print(f"sklearn     {sklearn.__version__}")
print(f"matplotlib  {matplotlib.__version__}")

DB_PATH = (Path.cwd().parent / "data" / "monsterlab.db").resolve()
assert DB_PATH.exists(), (
    f"Database not found at {DB_PATH} -- open this notebook so its working "
    "directory is the workbook/ folder (launch Jupyter from the repo root)"
)
# Read-only connection: the workbook never modifies the main database.
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
print(f"\nConnected read-only to {DB_PATH.name}")

In [ ]:
tables = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<12} {n:>5} rows")

expected = {"crosswalk", "fe_monsters", "sd_attacks", "sd_monsters", "sd_spells"}
missing = expected - set(tables)
assert not missing, f"Missing tables: {sorted(missing)} -- run the ingest pipeline first (python run_all.py)"
print("\nAll expected tables present.")

In [ ]:
import matplotlib.pyplot as plt

levels = pd.read_sql("SELECT level FROM sd_monsters", conn)["level"]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(levels, bins=range(0, int(levels.max()) + 2),
        color="#4269d0", edgecolor="white", linewidth=1)
ax.set_xlabel("Monster LV")
ax.set_ylabel("Monsters")
ax.set_title("Shadowdark monster levels — if you can read this, matplotlib works")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## The lifecycle map

Each notebook is one phase of the data-analytics lifecycle, run at small scale against
this repo's real data:

| # | Notebook | Lifecycle phase | You will |
|---|----------|-----------------|----------|
| 01 | `01_frame.ipynb` | Problem framing (business understanding) | Write the problem statement and three answerable questions |
| 02 | `02_acquire.ipynb` | Data acquisition | Pull the Open5e spells API with pagination and caching |
| 03 | `03_store.ipynb` | Storage & data management | Design a table, load it, query it with SQL |
| 04 | `04_clean.ipynb` | Cleaning & preparation | Parse messy attack strings and measure your success rate |
| 05 | `05_explore.ipynb` | Exploration (EDA) | Distributions, scatters, band means, a correlation matrix |
| 06 | `06_model.ipynb` | Modeling | Fit a regression, discover a leaky feature, fix it |
| 07 | `07_validate.ipynb` | Evaluation | Train/test split, residuals, rank vs linear correlation |
| 08 | `08_communicate.ipynb` | Communication | Write the findings memo for a non-technical reader |

Do them in order — each one leans on the one before it.